# SROIE (ICDAR 2019) — load & cache

Loads the [`jsdnrs/ICDAR2019-SROIE`](https://huggingface.co/datasets/jsdnrs/ICDAR2019-SROIE) dataset
(viewer-enabled, Parquet, CC-BY-4.0) straight from the Hub with pandas, inspects it, and saves
local copies under `data/sroie/` for the LayoutLMv3 pipeline.

Columns: `image`, `key`, `image_size`, `words`, `bboxes`, `entities` (company / date / address / total).

Run this notebook from the `data/` folder.

In [ ]:
# One-time setup — uncomment to install the dependencies:
# %pip install pandas pyarrow huggingface_hub pillow

In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
# Read the Parquet splits directly from the Hub (no loading script, no trust_remote_code).
splits = {
    "train": "data/train-00000-of-00001.parquet",
    "test": "data/test-00000-of-00001.parquet",
}
base = "hf://datasets/jsdnrs/ICDAR2019-SROIE/"

train_df = pd.read_parquet(base + splits["train"])
test_df = pd.read_parquet(base + splits["test"])

print(f"train: {train_df.shape}   test: {test_df.shape}")
train_df.columns.tolist()

In [ ]:
# Peek at one example: the KIE fields and the word/box arrays LayoutLMv3 needs.
row = train_df.iloc[0]
print("key:      ", row["key"])
print("entities: ", row["entities"])
print("n words:  ", len(row["words"]))
print("words[:8]:", row["words"][:8])
print("bboxes[0]:", row["bboxes"][0])

In [ ]:
# Cache the splits locally so later steps don't re-download.
out = Path("sroie")
out.mkdir(parents=True, exist_ok=True)

train_df.to_parquet(out / "train.parquet")
test_df.to_parquet(out / "test.parquet")

print("saved to", out.resolve())

In [ ]:
# Decode and save one receipt image as a sanity check.
from io import BytesIO
from PIL import Image

img_field = train_df.iloc[0]["image"]
img = Image.open(BytesIO(img_field["bytes"])) if isinstance(img_field, dict) else img_field
img.save(out / "sample.png")
print("sample image size:", img.size)

## Next: prep for LayoutLMv3

1. Turn each row's `entities` (company / date / address / total) into per-word **BIO labels** by
   matching the entity strings against `words` / `bboxes`.
2. Normalize `bboxes` to the 0–1000 range LayoutLMv3 expects.
3. Feed `words + bboxes + image` to `LayoutLMv3Processor`.

SROIE covers **date** and **total**; add CORD (Voxel51) for subtotal/tax, or DocILE for the full
invoice field set including invoice number.